In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# 1. PARÁMETROS DE LA SIMULACIÓN
# =====================================================================
SEM_CORTE = 20
PLAZO = 52
BAC_TOTAL = 4126638

# Metas globales auditadas para la Semana 20 (CPI 0.82 y SPI 0.86)
TARGET_PV = 1784771
TARGET_EV = 1537173
TARGET_AC = 1874601

# =====================================================================
# 2. DICCIONARIO WBS (122 Actividades de Subestación 115kV)
# =====================================================================
wbs_base = [
    ("1", "1. Ingeniería", "1.1", "1.1 Estudios Previos", ["Levantamiento Topografico", "Estudio de Suelos", "Estudio Resistividad", "Licencia Ambiental", "Aprobacion Trazado", "Estudio Arqueologico"], 1, 6, 0.01),
    ("1", "1. Ingeniería", "1.2", "1.2 Ingeniería Básica", ["Diagrama Unifilar", "Disposicion Planta", "Specs Trafo", "Cortocircuito", "Protecciones", "Basica Control"], 3, 10, 0.01),
    ("1", "1. Ingeniería", "1.3", "1.3 Detalle Civil", ["Cimentacion Trafo", "Porticos 115kV", "Drenajes", "Cajas y Ductos", "Edificio Control", "Memorias Civil", "Vias Acceso", "Cerramiento"], 5, 15, 0.015),
    ("1", "1. Ingeniería", "1.4", "1.4 Detalle Eléctrica", ["Malla a Tierra", "Conexionado", "Servicios Aux", "Control", "Cables", "Alumbrado", "Contra Incendio"], 8, 20, 0.015),

    ("2", "2. Obras Civiles", "2.1", "2.1 Movimiento Tierras", ["Descapote", "Excavacion Terrazas", "Relleno", "Vias Perimetrales", "Taludes", "Cerramiento Prov"], 5, 14, 0.04),
    ("2", "2. Obras Civiles", "2.2", "2.2 Cimentaciones", ["Zapatas Porticos", "Acero Porticos", "Vaciado Porticos", "Foso Trafo", "Muro Cortafuego", "Vaciado Foso", "Trampa Aceite", "Bases GIS"], 10, 25, 0.08),
    ("2", "2. Obras Civiles", "2.3", "2.3 Ductos y Drenajes", ["Excavacion Zanjas", "Tuberia PVC", "Cajas Inspeccion", "Sumideros", "Filtro Frances", "Concreto Ductos", "Relleno Zanjas", "Prueba Hidraulica"], 12, 28, 0.06),
    ("2", "2. Obras Civiles", "2.4", "2.4 Edificio Control", ["Cimentacion Edificio", "Vigas Cimentacion", "Mamposteria", "Losa Cubierta", "Pisos Falsos", "Aire Acondicionado", "Acabados", "Puertas"], 15, 35, 0.12),

    ("3", "3. Suministros", "3.1", "3.1 Equipos 115kV", ["Trafo 30MVA", "Interruptores 115kV", "Seccionadores", "TCs", "TTs", "Pararrayos", "Aisladores Soporte"], 1, 40, 0.25),
    ("3", "3. Suministros", "3.2", "3.2 Estructuras", ["Porticos 115kV", "Vigas Amarre", "Soportes Equipos", "Herrajes", "Pernos Anclaje"], 5, 30, 0.05),
    ("3", "3. Suministros", "3.3", "3.3 Control", ["Tableros Proteccion", "Tableros Control", "Celdas GIS 34.5kV", "Banco Baterias", "Cargadores", "SCADA", "RTU", "Medidores"], 5, 42, 0.10),
    ("3", "3. Suministros", "3.4", "3.4 Misceláneos", ["Cable Cobre Malla", "Cable XLPE", "Cable Control", "Conectores", "Terminales", "Bandejas Portacables"], 10, 45, 0.05),

    ("4", "4. Montaje", "4.1", "4.1 Malla a Tierra", ["Tendido Malla", "Soldaduras", "Colas Estructuras", "Medicion Resistencia", "Tratamiento Terreno"], 20, 30, 0.02),
    ("4", "4. Montaje", "4.2", "4.2 Equipos y Estructuras", ["Izaje Porticos", "Izaje Soportes", "Posicion Trafo", "Accesorios Trafo", "Montaje Interruptores", "Montaje Seccionadores", "Montaje TCs/TTs", "Alineacion", "Torqueo"], 30, 48, 0.05),
    ("4", "4. Montaje", "4.3", "4.3 Tendido y Conexionado", ["Potencia a Celdas", "Control a Edificio", "BT en Trafo", "Muflas", "Peinado Tableros", "Fibra Optica", "Bandejas"], 35, 50, 0.03),
    ("4", "4. Montaje", "4.4", "4.4 Salas Control", ["Fijacion Tableros", "Montaje GIS", "Banco Baterias", "Servicios Auxiliares", "Cableado Interpanel"], 38, 50, 0.02),

    ("5", "5. Pruebas", "5.1", "5.1 Pruebas SAT", ["Megado", "TTR", "Tiempos Cierre", "Resistencia Contactos", "Inyeccion Secundaria", "Descargas Parciales", "Rigidez Dielectrica"], 45, 51, 0.04),
    ("5", "5. Pruebas", "5.2", "5.2 Precomisionamiento", ["Wiring", "Logica Control", "Interbloqueos", "Señales SCADA", "Comunicaciones", "Lazos de Control"], 48, 52, 0.03),
    ("5", "5. Pruebas", "5.3", "5.3 Energización", ["Protocolos CND", "Inspeccion RETIE", "Energizacion Vacio", "Toma Carga", "Acta Entrega", "Capacitacion Operadores"], 51, 52, 0.01)
]

# Diccionarios de narrativa progresiva
nar_civ_op = ["Nivel freatico detectado. Inicia achique.", "Saturacion de terreno. 2 bombas extra.", "Achique continuo, frente a media marcha.", "Barro severo. Accesos limitados.", "Fundicion lenta por lodos en foso."]
nar_civ_adm = ["Costo directo elevado por ACPM.", "Aprobacion horas extras achique.", "Alquiler maquinaria impacta CPI.", "Mitigacion ambiental en curso.", "Alerta presupuestal en obra civil."]
nar_sum_op = ["Retraso en transito maritimo notificado.", "Inspeccion fisica aduanera en curso.", "Discrepancia documental en celdas.", "Equipos retenidos temporalmente.", "Reprogramando contratista de montaje."]
nar_sum_adm = ["Costos de bodegaje en puerto activos.", "Gestionando exencion de multas.", "SPI cae por no ingreso de equipos.", "Sobrecosto logistico acumulado.", "Retraso impacta facturacion del hito."]
nar_ok = ["Frente operativo.", "Ejecucion conforme."]

actividades = []
for cap_id, cap_name, sub_id, sub_name, lista_acts, ini, fin, peso_sub in wbs_base:
    peso_act = peso_sub / len(lista_acts)
    for i, act in enumerate(lista_acts):
        wbs_code = f"{sub_id}.{i+1}"
        actividades.append({
            "wbs": wbs_code, "cap": cap_name, "subcap": sub_name, "act": act,
            "ini": ini, "fin": fin, "bac": int(BAC_TOTAL * peso_act)
        })

# Corrección estricta: Inyección del remanente por redondeo
diff_bac = BAC_TOTAL - sum(a['bac'] for a in actividades)
actividades[0]['bac'] += diff_bac

# =====================================================================
# 3. MOTOR DE CÁLCULO EVM Y MATRIZ PLANA
# =====================================================================
rows = []
for act in actividades:
    pv_acc, ev_acc, ac_acc = 0, 0, 0
    duracion = act["fin"] - act["ini"] + 1
    pv_inc = act["bac"] / duracion

    for sem in range(1, PLAZO + 1):
        if act["ini"] <= sem <= act["fin"]: pv_acc = int(min(pv_acc + pv_inc, act["bac"]))
        elif sem > act["fin"]: pv_acc = act["bac"]

        coment_op, coment_adm = "", ""

        # Simulación hasta la Semana de Corte
        if sem <= SEM_CORTE:
            if sem >= act["ini"]:
                actual_spi = 0.96 if act["cap"].startswith("2") else (0.85 if act["cap"].startswith("3") else 1.0)
                actual_cpi = 0.76 if act["cap"].startswith("2") else (0.95 if act["cap"].startswith("3") else 1.0)

                if act["cap"].startswith("3") and sem >= 15: actual_spi = 0.65

                ev_inc_sem = pv_inc * actual_spi
                ac_inc_sem = ev_inc_sem / actual_cpi
                ev_acc = int(min(ev_acc + ev_inc_sem, act["bac"])) # Ensure EV does not exceed BAC for a single activity
                ac_acc = int(ac_acc + ac_inc_sem)

                if ev_acc >= act["bac"]:
                    coment_op, coment_adm = "Actividad terminada.", "Costo cerrado."
                else:
                    if act["cap"].startswith("2") and sem >= 12:
                        n_idx = (sem - 12) % len(nar_civ_op)
                        coment_op, coment_adm = nar_civ_op[n_idx], nar_civ_adm[n_idx]
                    elif act["cap"].startswith("3") and sem >= 15:
                        n_idx = (sem - 15) % len(nar_sum_op)
                        coment_op, coment_adm = nar_sum_op[n_idx], nar_sum_adm[n_idx]
                    else:
                        coment_op, coment_adm = nar_ok[0], nar_ok[1]
            else:
                ev_acc, ac_acc = 0, 0
                coment_op, coment_adm = "No iniciada", "No iniciada"

            rows.append([act["wbs"], act["cap"], act["subcap"], act["act"], sem, act["bac"], pv_acc, ev_acc, ac_acc, coment_op, coment_adm])
        else:
            # Semanas futuras (Para proyecciones PV)
            rows.append([act["wbs"], act["cap"], act["subcap"], act["act"], sem, act["bac"], pv_acc, "", "", "", ""])

df = pd.DataFrame(rows, columns=['WBS', 'Capitulo', 'Subcapitulo', 'Actividad', 'Semana', 'BAC', 'PV', 'EV', 'AC', 'Comentario_Operativo', 'Comentario_Administrativo'])

# =====================================================================
# 4. CALIBRACIÓN ABSOLUTA EN SEMANA 20
# =====================================================================
mask_20 = df["Semana"] == SEM_CORTE
diff_pv = TARGET_PV - df.loc[mask_20, "PV"].sum()
diff_ev = TARGET_EV - pd.to_numeric(df.loc[mask_20, "EV"]).sum()
diff_ac = TARGET_AC - pd.to_numeric(df.loc[mask_20, "AC"]).sum()

idx_activos = df[(mask_20) & (df["Capitulo"].str.startswith("2")) & (df["EV"] != 0)].index
if len(idx_activos) > 0:
    act_name = df.loc[idx_activos[0], "Actividad"]
    # Ajustar valor acumulado de esa actividad desde la semana de corte en adelante
    mask_ajuste = (df["Actividad"] == act_name) & (df["Semana"] == SEM_CORTE)
    df.loc[mask_ajuste, "PV"] += diff_pv
    df.loc[mask_ajuste, "EV"] = pd.to_numeric(df.loc[mask_ajuste, "EV"]) + diff_ev
    df.loc[mask_ajuste, "AC"] = pd.to_numeric(df.loc[mask_ajuste, "AC"]) + diff_ac

# Exportación del CSV (Solo datos históricos hasta la semana de corte para la matriz plana)
df_export = df[df["Semana"] <= SEM_CORTE].copy()
df_export.to_csv('data_proyecto.csv', sep=';', index=False)

# Auditoría Consola
df_20_final = df_export[df_export["Semana"] == SEM_CORTE].copy() # Fixed: Use df_export['Semana']
cpi = round(df_20_final["EV"].sum() / df_20_final["AC"].sum(), 2)
spi = round(df_20_final["EV"].sum() / df_20_final["PV"].sum(), 2)
print(f"--- SIMULACIÓN EJECUTADA ---")
print(f"BAC Total: {df_20_final['BAC'].sum()}")
print(f"Auditoría Sem 20 -> CPI: {cpi} | SPI: {spi}")
print(f"Archivo 'matriz_control_subestacion.csv' exportado.")

# =====================================================================
# 5. RENDERIZADO DE CURVAS S EN PANTALLA (MATPLOTLIB)
# =====================================================================
df['PV'] = pd.to_numeric(df['PV'], errors='coerce')
df['EV'] = pd.to_numeric(df['EV'], errors='coerce')
df['AC'] = pd.to_numeric(df['AC'], errors='coerce')

df_plot = df.groupby('Semana').agg({'PV': 'sum', 'EV': 'sum', 'AC': 'sum'}).reset_index()

# Ocultar ceros futuros para EV y AC en la gráfica
df_plot.loc[df_plot['Semana'] > SEM_CORTE, ['EV', 'AC']] = np.nan

plt.figure(figsize=(12, 6))
plt.plot(df_plot['Semana'], df_plot['PV'], label='Valor Planificado (PV)', color='#1f77b4', linestyle='--', linewidth=2)
plt.plot(df_plot['Semana'], df_plot['EV'], label='Valor Ganado (EV)', color='#2ca02c', linewidth=2.5)
plt.plot(df_plot['Semana'], df_plot['AC'], label='Costo Real (AC)', color='#d62728', linewidth=2.5)

plt.axvline(x=SEM_CORTE, color='gray', linestyle=':', label=f'Corte (Sem {SEM_CORTE})')

# Add labels to the end of the lines
last_sem = df_plot['Semana'].max()

# Label for PV with value
last_pv = df_plot.loc[df_plot['Semana'] == last_sem, 'PV'].iloc[0]
plt.text(last_sem - 0.5, last_pv, f'PV: ${last_pv:,.0f}', color='#1f77b4', va='center', ha='right', fontsize=10, fontweight='bold')

# Label for EV (at SEM_CORTE) with value
last_ev_sem_corte = df_plot.loc[df_plot['Semana'] == SEM_CORTE, 'EV'].iloc[0]
plt.text(SEM_CORTE + 0.5, last_ev_sem_corte, f'EV: ${last_ev_sem_corte:,.0f}', color='#2ca02c', va='center', ha='left', fontsize=10, fontweight='bold')

# Label for AC (at SEM_CORTE) with value
last_ac_sem_corte = df_plot.loc[df_plot['Semana'] == SEM_CORTE, 'AC'].iloc[0]
plt.text(SEM_CORTE + 0.5, last_ac_sem_corte - 0.05 * (df_plot['PV'].max()), f'AC: ${last_ac_sem_corte:,.0f}', color='#d62728', va='center', ha='left', fontsize=10, fontweight='bold')

# Label for the cut-off line
plt.text(SEM_CORTE, plt.ylim()[1] * 0.95, f'Corte (Sem {SEM_CORTE})', color='gray', ha='center', va='top', fontsize=10)

plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['AC'], where=(df_plot['Semana'] <= SEM_CORTE), color='red', alpha=0.1, label='Sobrecosto (Variación)')
plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['PV'], where=(df_plot['Semana'] <= SEM_CORTE), color='orange', alpha=0.1, label='Retraso (Variación)')

plt.title('Curva S de Desempeño - Subestación 115kV (Simulación)', fontsize=14, fontweight='bold')
plt.xlabel('Semanas del Proyecto', fontsize=11)
plt.ylabel('Inversión Acumulada (USD)', fontsize=11)
plt.xticks(np.arange(0, 53, step=4))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(loc='upper left')
plt.tight_layout()

# Mostrar Curvas
plt.show()